# Colab GPU Training Notebook (v2)

Changes from v1:
- 8 heads (dropped `density` — not a real pylidc field; dropped `internalStructure` — near-constant, no signal)
- 3D augmentation: random axis flips + 90-degree rotations on training patches only
- NaN-masked MSE loss so sparse annotation targets don't corrupt gradients
- Early stopping (patience=5) against avg val loss; 30-epoch ceiling
- Weight decay on Adam optimizer for regularization

Upload the LungInsight repository content to `/content/LungInsight` before
running (it must contain `cir_multihead_pipeline.py` and `se_resnet3d.py`).

In [ ]:
%pip install -q torch torchvision numpy pandas scikit-learn pylidc gradnorm-pytorch grad-cam

In [ ]:
import os
import sys

from google.colab import drive
drive.mount('/content/drive')

ROOT_DIR = '/content/LungInsight'
DRIVE_DIR = '/content/drive/MyDrive/LungInsight'
os.makedirs(DRIVE_DIR, exist_ok=True)

if not os.path.isdir(ROOT_DIR):
    raise RuntimeError(
        'Please upload repository content (including cir_multihead_pipeline.py '
        'and se_resnet3d.py) to /content/LungInsight before running this notebook.'
    )
if ROOT_DIR not in sys.path:
    sys.path.append(ROOT_DIR)

In [ ]:
import random
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.optim import Adam

from gradnorm_pytorch import GradNormLossWeighter

from cir_multihead_pipeline import create_multihead_model, FEATURE_NAMES, LIDCPatchDataset

## Augmentation

Applied to training patches only. Random flips along each of the three axes
and random 90-degree rotations in the axial plane are standard for 3D CT —
they are anatomically valid (nodules are approximately isotropic) and
effectively multiply the training set size without requiring extra DICOM
loading. No intensity augmentation is applied since the patches are already
in HU and the model relies on HU values for calcification detection.

In [ ]:
def augment_patch(tensor: torch.Tensor) -> torch.Tensor:
    """Random flips and 90-degree in-plane rotations for a (1, D, H, W) tensor."""
    # Random flip along each spatial axis independently
    for dim in [1, 2, 3]:  # D, H, W (axis 0 is channel)
        if random.random() > 0.5:
            tensor = torch.flip(tensor, dims=[dim])
    # Random 90-degree rotation in the axial (H, W) plane: 0, 90, 180, or 270 degrees
    k = random.randint(0, 3)
    if k > 0:
        tensor = torch.rot90(tensor, k=k, dims=[2, 3])  # H, W axes
    return tensor

## Config and data loaders

In [ ]:
TRAIN_CSV = '/content/drive/MyDrive/LungInsight/cpu_split/train_split.csv'
VAL_CSV   = '/content/drive/MyDrive/LungInsight/cpu_split/val_split.csv'
BATCH_SIZE = 4
EPOCHS = 30
PATIENCE = 5       # early stopping: halt if val loss doesn't improve for this many epochs
LR = 1e-4
WEIGHT_DECAY = 1e-4
GRADNORM_LR = 1e-4
RESTORING_FORCE_ALPHA = 0.12  # mild restoring force; 0.0 = perfectly equal weights
NUM_WORKERS = 2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
print('Heads:', FEATURE_NAMES)

train_dataset = LIDCPatchDataset(TRAIN_CSV, device='cpu')
val_dataset   = LIDCPatchDataset(VAL_CSV,   device='cpu')
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f'Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}')

## Model, optimizer, and GradNorm loss weighter

In [ ]:
model = create_multihead_model(head_names=FEATURE_NAMES, device=device)
optimizer = Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

gradnorm_parameter = model.layer4[-1].conv3.weight

loss_weighter = GradNormLossWeighter(
    num_losses=len(FEATURE_NAMES),
    learning_rate=GRADNORM_LR,
    restoring_force_alpha=RESTORING_FORCE_ALPHA,
    grad_norm_parameters=gradnorm_parameter,
)

## Training and validation loops

NaN masking: targets with no annotator rating are NaN in the manifest.
Passing NaN into MSE produces NaN loss which corrupts all GradNorm weights.
Each head's loss is computed only over the batch samples that have a valid
(non-NaN) target. If an entire batch has no valid targets for a head, a
zero-loss placeholder with a grad_fn is used so GradNorm doesn't error.

In [ ]:
def masked_mse(pred: torch.Tensor, tgt: torch.Tensor) -> torch.Tensor:
    """MSE over non-NaN targets only. Returns a zero-grad tensor if all NaN."""
    valid = ~torch.isnan(tgt)
    if valid.any():
        return F.mse_loss(pred[valid], tgt[valid])
    return pred.mean() * 0.0  # preserves grad_fn, contributes nothing


def train_one_epoch(loader, model, loss_weighter, optimizer, device):
    model.train()
    running = {feat: 0.0 for feat in FEATURE_NAMES}
    n_batches = 0

    for patches, targets in loader:
        patches = augment_patch(patches).to(device)
        targets = {k: v.to(device) for k, v in targets.items()}

        optimizer.zero_grad()
        outputs = model(patches)

        losses = [masked_mse(outputs[feat], targets[feat]) for feat in FEATURE_NAMES]
        loss_weighter.backward(losses)
        optimizer.step()

        for feat, l in zip(FEATURE_NAMES, losses):
            running[feat] += l.item()
        n_batches += 1

    return {feat: running[feat] / max(n_batches, 1) for feat in FEATURE_NAMES}


def validate_one_epoch(loader, model, device):
    model.eval()
    running = {feat: 0.0 for feat in FEATURE_NAMES}
    n_batches = 0

    with torch.no_grad():
        for patches, targets in loader:
            patches = patches.to(device)
            targets = {k: v.to(device) for k, v in targets.items()}
            outputs = model(patches)
            for feat in FEATURE_NAMES:
                running[feat] += masked_mse(outputs[feat], targets[feat]).item()
            n_batches += 1

    return {feat: running[feat] / max(n_batches, 1) for feat in FEATURE_NAMES}

## Run training with early stopping

In [ ]:
best_val_loss = float('inf')
epochs_without_improvement = 0
best_model_path = os.path.join(DRIVE_DIR, 'best_model_gpu.pth')

for epoch in range(1, EPOCHS + 1):
    print(f'=== Epoch {epoch}/{EPOCHS} ===')
    train_loss = train_one_epoch(train_loader, model, loss_weighter, optimizer, device)
    val_loss   = validate_one_epoch(val_loader, model, device)

    print('Training losses:  ', {k: round(v, 4) for k, v in train_loss.items()})
    print('Validation losses:', {k: round(v, 4) for k, v in val_loss.items()})

    avg_val = sum(val_loss.values()) / len(val_loss)
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        epochs_without_improvement = 0
        torch.save(model.state_dict(), best_model_path)
        print(f'Saved best model (avg val loss {best_val_loss:.4f})')
    else:
        epochs_without_improvement += 1
        print(f'No improvement ({epochs_without_improvement}/{PATIENCE})')
        if epochs_without_improvement >= PATIENCE:
            print(f'Early stopping at epoch {epoch}.')
            break

print(f'Training complete. Best avg val loss: {best_val_loss:.4f}')